# DataPy · Übungsset für die Wiederholung: Lösungen

Erst nach einem eigenen Versuch ansehen. Alle Lösungen bestehen die Testzellen.

In [2]:
import os
import numpy as np
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

TMP = "uebung_tmp"
os.makedirs(TMP, exist_ok=True)

def write(name, text):
    """Hilfsfunktion für die Tests: schreibt eine Textdatei in den Übungsordner."""
    path = os.path.join(TMP, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

def out(name):
    return os.path.join(TMP, name)


## W1 · Warm-up: Was kommt raus?

Trage für jeden Ausdruck die **Form des Ergebnisses** als Tupel ein, z. B. `(3, 4)`, oder den String `"Error"`, wenn ein `ValueError` entsteht. Erst überlegen, dann die Testzelle ausführen. (Frage 2 im Test war genau dieser Typ.)

```python
a) np.ones((3, 4)) + np.arange(4)
b) np.ones((3, 4)) + np.arange(3)
c) np.ones((3, 4)) + np.arange(3).reshape((-1, 1))
d) np.ones((2, 3)) * np.array([10, 100, 1000])
e) np.ones((2, 3)) / np.ones((2, 3)).sum(axis=1)
f) np.arange(3).reshape((3, 1)) + np.arange(4)
g) np.ones((3, 4)).sum(axis=0)
h) np.ones((3, 4)).mean(axis=1)
```

In [3]:
antworten = {
    "a": (3, 4),      # 4 und 4 passen
    "b": "Error",     # 4 und 3 passen nicht
    "c": (3, 4),      # (3, 4) und (3, 1): 4/1 ok, 3/3 ok
    "d": (2, 3),      # 3 und 3 passen
    "e": "Error",     # (2, 3) und (2,): 3 und 2 passen nicht
    "f": (3, 4),      # (3, 1) und (4,): beide werden gedehnt
    "g": (4,),        # axis=0 verschwindet
    "h": (3,),        # axis=1 verschwindet
}

In [4]:
_ausdruecke = {
    "a": lambda: np.ones((3, 4)) + np.arange(4),
    "b": lambda: np.ones((3, 4)) + np.arange(3),
    "c": lambda: np.ones((3, 4)) + np.arange(3).reshape((-1, 1)),
    "d": lambda: np.ones((2, 3)) * np.array([10, 100, 1000]),
    "e": lambda: np.ones((2, 3)) / np.ones((2, 3)).sum(axis=1),
    "f": lambda: np.arange(3).reshape((3, 1)) + np.arange(4),
    "g": lambda: np.ones((3, 4)).sum(axis=0),
    "h": lambda: np.ones((3, 4)).mean(axis=1),
}
_falsch = 0
for k, f in _ausdruecke.items():
    try:
        richtig = f().shape
    except ValueError:
        richtig = "Error"
    if antworten[k] == richtig:
        print(f"✓ {k}")
    else:
        _falsch += 1
        print(f"✗ {k}: deine Antwort {antworten[k]}, richtig ist {richtig}")
assert _falsch == 0, f"{_falsch} falsch"
print("✓ W1 bestanden")

✓ a
✓ b
✓ c
✓ d
✓ e
✓ f
✓ g
✓ h
✓ W1 bestanden


## P1 · CSV bereinigen

Schreibe `clean_csv(input_path, output_path)`:

1. CSV einlesen.
2. Doppelte Zeilen entfernen und alle Zeilen, in denen **irgendein** Wert fehlt.
3. Ergebnis ohne DataFrame-Index unter `output_path` speichern.
4. Die Anzahl der verbleibenden Zeilen als `int` zurückgeben.

In [5]:
def clean_csv(input_path, output_path):
    df = pd.read_csv(input_path)
    df = df.drop_duplicates()
    df = df.dropna()
    df.to_csv(output_path, index=False)
    return len(df)

In [6]:
p = write("p1.csv", "name,alter\nAda,36\nBen,\nAda,36\nCem,29\n")
n = clean_csv(p, out("p1_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("p1_out.csv"), encoding="utf-8").read().splitlines() == ["name,alter", "Ada,36.0", "Cem,29.0"]
print("✓ P1 bestanden")

✓ P1 bestanden


## P2 · Umsatz je Produkt

Eine CSV-Datei hat die Spalten `produkt`, `datum`, `umsatz`. Einige Umsätze fehlen, Zeilen können doppelt vorkommen. Schreibe `sales_report(input_path, output_path)`:

1. Einlesen, doppelte Zeilen entfernen, Zeilen ohne `umsatz` entfernen.
2. Je Produkt die **Summe** des Umsatzes (auf 2 Nachkommastellen gerundet) und die **Anzahl** der Verkäufe berechnen.
3. Als CSV speichern mit den Spalten `produkt`, `summe`, `anzahl` in dieser Reihenfolge, sortiert nach `summe` **absteigend**, ohne Index.
4. Den Namen des Produkts mit dem höchsten Umsatz als `str` zurückgeben. Bleiben keine Zeilen übrig: nur die Kopfzeile schreiben und `None` zurückgeben.

In [7]:
def sales_report(input_path, output_path):
    df = pd.read_csv(input_path)
    df = df.drop_duplicates()
    df = df.dropna(subset=['umsatz'])
    result = df.groupby('produkt')['umsatz'].agg(['sum', 'count'])
    result = result.rename(columns={'sum': 'summe', 'count': 'anzahl'})
    result['summe'] = result['summe'].round(2)
    result = result.sort_values(by='summe', ascending=False).reset_index()
    result.to_csv(output_path, index=False)
    if len(result) == 0:
        return None
    return result['produkt'].iloc[0]

In [8]:
p = write("p2.csv", "produkt,datum,umsatz\nTee,2026-10-01,3.5\nKaffee,2026-10-01,2.8\nKaffee,2026-10-01,2.8\nKaffee,2026-10-02,2.81\nTee,2026-10-02,\nKakao,2026-10-03,4.0\n")
best = sales_report(p, out("p2_out.csv"))
assert best == "Kaffee", best
assert open(out("p2_out.csv"), encoding="utf-8").read().splitlines() == ["produkt,summe,anzahl", "Kaffee,5.61,2", "Kakao,4.0,1", "Tee,3.5,1"], open(out("p2_out.csv")).read()
p = write("p2_leer.csv", "produkt,datum,umsatz\nTee,2026-10-02,\n")
assert sales_report(p, out("p2_leer_out.csv")) is None
assert open(out("p2_leer_out.csv"), encoding="utf-8").read().strip() == "produkt,summe,anzahl"
print("✓ P2 bestanden")

✓ P2 bestanden


## P3 · Zwei Dateien verbinden

`studierende.csv` hat die Spalten `name`, `kurs_id`. `kurse.csv` hat `kurs_id`, `kurs`. Schreibe `join_courses(students_path, courses_path, output_path)`:

1. Beide Dateien einlesen.
2. So verbinden, dass **alle Studierenden** erhalten bleiben, auch ohne passenden Kurs.
3. Fehlende Kursnamen durch den Text `"unbekannt"` ersetzen.
4. Mit den Spalten `name`, `kurs` (nur diese, in dieser Reihenfolge), sortiert nach `name`, ohne Index speichern.
5. Die Anzahl der Studierenden **ohne** passenden Kurs als `int` zurückgeben.

In [9]:
def join_courses(students_path, courses_path, output_path):
    students = pd.read_csv(students_path)
    courses = pd.read_csv(courses_path)
    df = pd.merge(students, courses, on='kurs_id', how='left')
    n_missing = int(df['kurs'].isna().sum())
    df['kurs'] = df['kurs'].fillna('unbekannt')
    df = df[['name', 'kurs']].sort_values(by='name')
    df.to_csv(output_path, index=False)
    return n_missing

In [10]:
s = write("p3_s.csv", "name,kurs_id\nCem,3\nAda,1\nBen,2\nDia,9\n")
c = write("p3_c.csv", "kurs_id,kurs\n1,DataPy\n2,HCI\n")
n = join_courses(s, c, out("p3_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("p3_out.csv"), encoding="utf-8").read().splitlines() == ["name,kurs", "Ada,DataPy", "Ben,HCI", "Cem,unbekannt", "Dia,unbekannt"]
print("✓ P3 bestanden")

✓ P3 bestanden


## P4 · Noten auswerten (ohne Dateien)

Gegeben ist ein DataFrame mit den Spalten `name`, `semester`, `note`. Schreibe `grade_stats(df)`. Die Funktion gibt ein **Dictionary** zurück mit:

- `"bestanden"`: Anzahl der Zeilen mit `note <= 4.0` (`int`)
- `"beste"`: Name der Person mit der besten (kleinsten) Note (`str`)
- `"schnitt"`: Dictionary `{semester: Mittelwert der Note}`, auf 2 Nachkommastellen gerundet

Zeilen ohne Note werden vorher entfernt. Das übergebene `df` darf **nicht** verändert werden.

In [11]:
def grade_stats(df):
    d = df.dropna(subset=['note'])            # dropna gibt einen neuen DataFrame zurück
    bestanden = int((d['note'] <= 4.0).sum())
    beste = d.sort_values(by='note')['name'].iloc[0]
    schnitt = d.groupby('semester')['note'].mean().round(2).to_dict()
    return {'bestanden': bestanden, 'beste': beste, 'schnitt': schnitt}

In [12]:
_df = pd.DataFrame({"name": ["Ada", "Ben", "Cem", "Dia", "Eli"],
                    "semester": [1, 1, 3, 3, 3],
                    "note": [1.3, 5.0, 2.0, np.nan, 2.7]})
_vorher = _df.copy()
r = grade_stats(_df)
assert r["bestanden"] == 3, r
assert r["beste"] == "Ada", r
assert r["schnitt"] == {1: 3.15, 3: 2.35}, r
assert _df.equals(_vorher), "Original wurde verändert"
print("✓ P4 bestanden")

✓ P4 bestanden


## N1 · Zeilen und Spalten normieren

Schreibe zwei Funktionen, ohne Schleife:

- `center_columns(M)`: zieht von jeder **Spalte** ihren Mittelwert ab (danach ist jeder Spaltenmittelwert 0).
- `scale_rows(M)`: teilt jede **Zeile** durch ihr Maximum (danach ist das Maximum jeder Zeile 1).

Überlege vorher: Bei welcher der beiden brauchst du `reshape((-1, 1))` und warum?

In [13]:
def center_columns(M):
    return M - M.mean(axis=0)                    # (2, 3) - (3,): passt direkt

def scale_rows(M):
    return M / M.max(axis=1).reshape((-1, 1))    # (2, 3) / (2, 1): Spaltenvektor nötig

In [14]:
_M = np.array([[1., 2., 4.], [3., 6., 8.]])
assert np.allclose(center_columns(_M), [[-1, -2, -2], [1, 2, 2]])
assert np.allclose(scale_rows(_M), [[0.25, 0.5, 1.0], [0.375, 0.75, 1.0]])
print("✓ N1 bestanden")

✓ N1 bestanden


## N2 · Messwerte bereinigen

`data` ist ein 2D-Array: jede Zeile ist ein Sensor, jede Spalte eine Messung. Schreibe `clean_sensors(data, low, high)`:

1. Eine **Kopie** anlegen (das Original bleibt unverändert).
2. Alle Werte kleiner als `low` oder größer als `high` durch `np.nan` ersetzen.
3. Ein Tupel zurückgeben: `(Mittelwert je Sensor ohne NaN, Index des Sensors mit dem höchsten Mittelwert als int)`.

In [15]:
def clean_sensors(data, low, high):
    d = data.copy()
    d[(d < low) | (d > high)] = np.nan
    means = np.nanmean(d, axis=1)
    return means, int(np.argmax(means))

In [16]:
_d = np.array([[20., 21., 999.], [18., -50., 22.], [25., 26., 27.]])
_vorher = _d.copy()
means, best = clean_sensors(_d, 0, 100)
assert np.allclose(means, [20.5, 20.0, 26.0]), means
assert best == 2 and isinstance(best, int)
assert np.array_equal(_d, _vorher), "Original wurde verändert"
print("✓ N2 bestanden")

✓ N2 bestanden


## B1 · Links in eine CSV-Datei

Schreibe `links_to_csv(html, output_path)`:

1. Alle `<a>`-Tags finden, die ein `href` haben.
2. Daraus einen DataFrame mit den Spalten `text` (Linktext ohne Leerraum am Rand) und `href` bauen.
3. Ohne Index als CSV speichern.
4. Die Anzahl der Links als `int` zurückgeben.

In [17]:
def links_to_csv(html, output_path):
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for a in soup.find_all('a'):
        if a.get('href'):
            rows.append({'text': a.get_text().strip(), 'href': a.get('href')})
    df = pd.DataFrame(rows, columns=['text', 'href'])
    df.to_csv(output_path, index=False)
    return len(df)

In [18]:
_html = '<p><a href="/a"> Eins </a> <a>ohne Ziel</a> <a href="https://bht-berlin.de">BHT</a></p>'
n = links_to_csv(_html, out("b1_out.csv"))
assert n == 2, n
assert open(out("b1_out.csv"), encoding="utf-8").read().splitlines() == ["text,href", "Eins,/a", "BHT,https://bht-berlin.de"]
assert links_to_csv("<p>nichts</p>", out("b1_leer.csv")) == 0
assert open(out("b1_leer.csv"), encoding="utf-8").read().strip() == "text,href"
print("✓ B1 bestanden")

✓ B1 bestanden


## B2 · Seite zusammenfassen

Schreibe `page_summary(html)`. Die Funktion gibt ein Dictionary zurück:

- `"title"`: Text des `<title>`-Tags, oder `None`, wenn es keinen gibt
- `"h1"`: Text der ersten `<h1>`, oder `None`
- `"n_p"`: Anzahl der `<p>`-Tags
- `"items"`: Liste mit den Texten aller `<li>`-Elemente (ohne Leerraum am Rand)

Achtung: `soup.find(...)` liefert `None`, wenn es nichts findet. `None.get_text()` ist ein `AttributeError`.

In [19]:
def page_summary(html):
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.find('title')
    h1 = soup.find('h1')
    return {
        'title': title.get_text().strip() if title else None,
        'h1': h1.get_text().strip() if h1 else None,
        'n_p': len(soup.find_all('p')),
        'items': [li.get_text().strip() for li in soup.find_all('li')],
    }

In [20]:
_html = "<html><head><title>Mensa</title></head><body><p>Hallo</p><ul><li> Curry </li><li>Suppe</li></ul><p>Tschüss</p></body></html>"
assert page_summary(_html) == {"title": "Mensa", "h1": None, "n_p": 2, "items": ["Curry", "Suppe"]}, page_summary(_html)
assert page_summary("<h1>Nur Überschrift</h1>") == {"title": None, "h1": "Nur Überschrift", "n_p": 0, "items": []}
print("✓ B2 bestanden")

✓ B2 bestanden


## B3 · HTML-Tabelle auswerten

Ein HTML-Text enthält eine Tabelle mit den Spalten `Bezirk` und `Einwohner`. Schreibe `biggest_district(html)`:

1. Die erste Tabelle als DataFrame einlesen (`pd.read_html` braucht `StringIO(html)` und liefert eine **Liste**).
2. Den Namen des Bezirks mit den meisten Einwohnern als `str` zurückgeben.

In [21]:
def biggest_district(html):
    df = pd.read_html(StringIO(html))[0]
    df = df.sort_values(by='Einwohner', ascending=False)
    return df['Bezirk'].iloc[0]

In [22]:
_html = "<table><tr><th>Bezirk</th><th>Einwohner</th></tr><tr><td>Mitte</td><td>332100</td></tr><tr><td>Pankow</td><td>368956</td></tr><tr><td>Spandau</td><td>225420</td></tr></table>"
assert biggest_district(_html) == "Pankow"
print("✓ B3 bestanden")

✓ B3 bestanden


## M1 · Probeklausur-Aufgabe 1 (20 Minuten, ohne Werkzeugkasten)

**Stell einen Timer auf 20 Minuten und scrolle nicht nach oben.**

Eine CSV-Datei enthält Prüfungsergebnisse mit den Spalten `name`, `kurs` und `punkte`. Einige Punktwerte fehlen, außerdem können Zeilen doppelt vorkommen. Implementiere `summarize_exams(input_path, output_path)` mit pandas:

1. CSV-Datei einlesen, doppelte Zeilen sowie alle Zeilen ohne `punkte` entfernen.
2. Für jeden Kurs den Mittelwert der Punkte (auf eine Nachkommastelle gerundet), die höchste Punktzahl und die Anzahl der Ergebnisse bestimmen.
3. Als CSV unter `output_path` speichern, mit den Spalten `kurs`, `mean_punkte`, `max_punkte`, `n` in dieser Reihenfolge. Die Kurse müssen alphabetisch sortiert sein. Der Index darf nicht mitgespeichert werden.
4. Die Anzahl der Kurse als `int` zurückgeben.

Sonderfall: Bleiben keine Ergebnisse übrig, wird eine Datei nur mit der Kopfzeile geschrieben und `0` zurückgegeben.

In [23]:
def summarize_exams(input_path, output_path):
    df = pd.read_csv(input_path)
    df = df.drop_duplicates()
    df = df.dropna(subset=['punkte'])
    result = df.groupby('kurs')['punkte'].agg(['mean', 'max', 'count'])
    result = result.rename(columns={'mean': 'mean_punkte', 'max': 'max_punkte', 'count': 'n'})
    result['mean_punkte'] = result['mean_punkte'].round(1)
    result = result.sort_index().reset_index()
    result.to_csv(output_path, index=False)
    return len(result)

In [24]:
p = write("m1.csv", "name,kurs,punkte\nAda,HCI,40\nBen,DataPy,31\nBen,DataPy,31\nCem,DataPy,36\nDia,HCI,\nEli,DataPy,38\n")
n = summarize_exams(p, out("m1_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("m1_out.csv"), encoding="utf-8").read().splitlines() == ["kurs,mean_punkte,max_punkte,n", "DataPy,35.0,38.0,3", "HCI,40.0,40.0,1"], open(out("m1_out.csv")).read()
p = write("m1_leer.csv", "name,kurs,punkte\nDia,HCI,\n")
assert summarize_exams(p, out("m1_leer_out.csv")) == 0
assert open(out("m1_leer_out.csv"), encoding="utf-8").read().strip() == "kurs,mean_punkte,max_punkte,n"
print("✓ M1 bestanden")

✓ M1 bestanden


## M2 · Probeklausur-Aufgabe 2 (20 Minuten, ohne Werkzeugkasten)

**Timer auf 20 Minuten.** Diesmal mit Beautiful Soup statt CSV-Eingabe.

Ein HTML-Text enthält Produkte in dieser Form:

```html
<div class="produkt"><h2>Tee</h2><span>3,50 €</span></div>
```

Implementiere `products_to_csv(html, output_path)`:

1. Alle `<h2>`-Tags (Produktnamen) und alle `<span>`-Tags (Preise) auslesen. Beide Listen sind gleich lang und in derselben Reihenfolge.
2. Die Preise in `float` umwandeln (`"3,50 €"` → `3.5`).
3. Einen DataFrame mit den Spalten `produkt` und `preis` bauen, nach `preis` aufsteigend sortieren und ohne Index als CSV speichern.
4. Den Durchschnittspreis, auf 2 Nachkommastellen gerundet, als `float` zurückgeben. Gibt es keine Produkte: nur die Kopfzeile schreiben und `0.0` zurückgeben.

In [25]:
def products_to_csv(html, output_path):
    soup = BeautifulSoup(html, 'html.parser')
    names = [h.get_text().strip() for h in soup.find_all('h2')]
    prices = [float(s.get_text().replace('€', '').strip().replace(',', '.'))
              for s in soup.find_all('span')]
    df = pd.DataFrame({'produkt': names, 'preis': prices})
    df = df.sort_values(by='preis')
    df.to_csv(output_path, index=False)
    if len(df) == 0:
        return 0.0
    return float(round(df['preis'].mean(), 2))

In [26]:
_html = '<div class="produkt"><h2>Tee</h2><span>3,50 €</span></div><div class="produkt"><h2> Kaffee </h2><span>2,80 €</span></div><div class="produkt"><h2>Kakao</h2><span>4,05 €</span></div>'
m = products_to_csv(_html, out("m2_out.csv"))
assert m == 3.45 and isinstance(m, float), m
assert open(out("m2_out.csv"), encoding="utf-8").read().splitlines() == ["produkt,preis", "Kaffee,2.8", "Tee,3.5", "Kakao,4.05"]
assert products_to_csv("<p>leer</p>", out("m2_leer.csv")) == 0.0
assert open(out("m2_leer.csv"), encoding="utf-8").read().strip() == "produkt,preis"
print("✓ M2 bestanden")

✓ M2 bestanden
